# School Absence and Parental Crime
## Preparing Data

This notebook explains the data collection run in `/scripts/collect_data.py` step-by-step.

For this notebook to work, make sure the project is installed *including optional dependencies*:

In the terminal:
```
pip install -e .[notebooks]
```

In [1]:
from pathlib import Path
import pickle

import pandas as pd

from absence_and_crime.utils.stata_io import fetch, gather

In [2]:
from absence_and_crime.config import PATHS
paths = PATHS

### Cleaning Attendance Records

We gather attendance records for the selected years and restrict them as follows:
- Keep only records from regular public schools, grades 0.-9.
- Keep only children observed since 0. grade

We also remove any double registrations and deal with invalid values.

The cleaned data is stored in `temp/absence.parquet`.

In [3]:
# list of recorded school years ('11-'15 cover 2 years each)
years = [2011, 2013, 2015, 2017, 2018, 2019, 2020, 2021, 2022, 2023]

# paths to data sets 
p1 = paths.data.dst_raw / Path(
    r"Eksterne data\STIL\Trivsel_fravaer_nov_2018"
)  # 2011-2017, 2019
p2 = paths.data.dst_raw / Path(r"Eksterne data\STIL 2023")  # 2021-2022
p3 = paths.data.dst_raw / Path(r"Eksterne data\STIL_2024")  # 2018, 2020, 2023

In [4]:
# Runs only if absence.parquet does not already exist in temp/
if not Path(paths.temp / "absence.parquet").exists():

    absence = {}
    for y in years:

        # ---- Load ----

        if y in [2011, 2013, 2015]:
            ny = y + 1 - 2000
            df = pd.read_sas(p1 / f"fravaer_{y}_{ny}.sas7bdat", format="sas7bdat")
        elif y in [2017, 2019]:
            df = pd.read_sas(p1 / f"fravaer_{y}.sas7bdat", format="sas7bdat")
        elif y in [2018, 2020, 2023]:
            df = pd.read_sas(p3 / f"fravaer_{y}.sas7bdat", format="sas7bdat")
        elif y in [2021, 2022]:
            df = pd.read_sas(p2 / f"afid_fravaer_{y}.sas7bdat", format="sas7bdat")
        else:
            raise ValueError(f"Unexpected year: {y}")

        # ---- Basic Editing ----

        # Renaming variables
        df.columns = df.columns.str.lower()

        if "cprnummer" in df.columns:
            df.rename(columns={"cprnummer": "pnr"}, inplace=True)

        # Only regular schools, grades 0.-9.
        df = df[
            (df["insttype"].str.decode("utf-8") == "1012")
            & (df["klassetrin"] >= 0)
            & (df["klassetrin"] <= 9)
        ]

        # Define absenteeism
        df["abs_rate"] = (df["dageialtfra"] / df["dageaktiv"] * 100).clip(
            lower=0, upper=100
        )

        # Selecting variables
        vl = ["pnr", "maaned", "klassetrin", "abs_rate"]
        df = df[vl]

        # Making dates monthly
        df["maaned"] = pd.to_datetime(df["maaned"], format="%Y%m").dt.to_period("M")

        # Destringing
        df["pnr"] = pd.to_numeric(df["pnr"])

        absence[y] = df
        del df

    # Combine all years
    absence = pd.concat(absence.values(), ignore_index=True)

    # Make child-month observations unique
    absence = (
        absence.rename(columns={"maaned": "month", "klassetrin": "grade"})
        .groupby(["pnr", "month"], as_index=False)
        .agg({"grade": "last", "abs_rate": "max"})
    )

    # Limit to children observed since grade 0
    absence = absence.loc[
        lambda d: d["grade"].eq(0).groupby(d["pnr"]).transform("any")
    ]

    # Limit observation period
    absence = absence[absence["month"] <= pd.Period("2021-06", freq="M")]

    # Save
    absence.to_parquet(paths.temp / "absence.parquet", index=False)

else:
    absence = pd.read_parquet(paths.temp / "absence.parquet")

### Defining Families

We find the children and their parents in the yearly population registers.

Child-parent links are allowed to change over time. We record the beginning and end year of these links.

We also ensure that the parents were found in the population register themselves. 

The resulting data of unique child-parent pairs (incl. the period of their relationship and basic demographic variables) is saved in `temp/families.parquet`.

In [5]:
# Runs only if families.parquet does not already exist in temp/
if not Path(paths.temp / "families.parquet").exists():

    # List of children and their first month of school
    ids = absence.sort_values("month").drop_duplicates("pnr", keep="first")[
        ["pnr", "month"]
    ]

    # Find children and their parents
    families = gather(
        paths.data.dst_raw,
        names=range(2003, 2023),
        file_pattern="bef12_{name}.dta",
        add_name="year",
        concatenate=True,
        filters={"pnr": ids["pnr"]},
        columns=["pnr", "koen", "foed_dag", "ie_type", "mor_id", "far_id"],
    )

    # Set time-invariant variables to latest recorded value
    vl = ["koen", "foed_dag", "ie_type"]
    families = families.sort_values(["pnr", "year"])
    families[vl] = families.groupby("pnr")[vl].transform("last")

    # ---- Crop to appropriate subset of children

    # Only years prior to year of 17th birthday
    families = families.loc[families["year"] - families["foed_dag"].dt.year <= 16]

    # Must have started school at ages 5-7
    families = families.merge(
        ids, on="pnr", how="left"
    ).loc[  # add first month of school
        lambda d: (d.month.dt.year - d.foed_dag.dt.year).between(5, 7)
    ]

    # Reshaping: make long and drop missing parents
    families = families.melt(
        id_vars=["pnr", "koen", "foed_dag", "ie_type", "year", "month"],
        value_vars=["mor_id", "far_id"],
        var_name="mom",
        value_name="parent",
    ).loc[lambda d: d["parent"] != ""]

    # Destring parent IDs
    families["parent"] = pd.to_numeric(families["parent"]).astype("int64")

    # Mom indicator
    families["mom"] = (families["mom"] == "mor_id").astype("uint8")

    # Time of parenthood
    families["parent_start"] = families.groupby(["pnr", "parent"])["year"].transform(
        "min"
    )
    families["parent_end"] = families.groupby(["pnr", "parent"])["year"].transform(
        "max"
    )

    # Remove time dimension and make child-parent pair unique
    families.drop(columns=["year"], inplace=True)
    families.drop_duplicates(inplace=True)

    # Minor edits
    families = families.assign(
        koen=lambda d: d.koen.eq(2).astype(int),  # female dummy
        ie_type=lambda d: d.ie_type.ne(1).astype(int),  # immigrant/descendant dummy
    ).rename(
        columns={
            "month": "start_of_school",
            "foed_dag": "date_of_birth",
            "koen": "female",
            "ie_type": "non_danish",
        }
    )

    # ---- Find parents

    ids = set(families["parent"])

    ps = gather(
        paths.data.dst_raw,
        names=range(2003, 2023),
        file_pattern="bef12_{name}.dta",
        add_name="year",
        concatenate=True,
        filters={"pnr": ids},
        columns=["pnr", "koen"],
    )

    # Latest registered sex
    ps = (
        ps.sort_values(["pnr", "year"])
        .drop_duplicates("pnr", keep="last")[["pnr", "koen"]]
        .rename(columns={"pnr": "parent"})
        .reset_index(drop=True)
    )

    # Add to child records
    families = (
        families.merge(ps, on="parent", how="inner", validate="m:1")
        .assign(mom=lambda d: d.koen.eq(2).where(d.koen.notna()))
        .drop(columns=["koen"])
        .drop_duplicates()
    )

    # Save
    families.to_parquet(paths.temp / "families.parquet", index=False)

else:
    families = pd.read_parquet(paths.temp / "families.parquet")

### Collecting Criminal Records

We collect all recorded criminal history on the parents, herunder:
- Penal-law criminal offenses and charges (jointly defined; no charge, no crime)
- Penal-law convictions
- Arrests
- Non-arrest incarcerations (pre-trial detentian and time served)

These are saved in a dictionary of DataFrames in `temp/records.pkl`.

Most of the records are straightforward, but non-arrest incarcerations require some work:
1. We drop older records with missing release dates (release dates are only systematically recorded since 1991); these should be too old to matter in our case anyway.
2. We assume that missing release dates on transfer records are the same as entry-dates (i.e., transfer of prisoners should not take multiple days).
3. A few records still have missing release dates; most of these are from fairly recent incarcerations, so we assume that the incarceration is ongoing. 

In [6]:
from absence_and_crime.transforms import make_monthly, merge_spells

In [7]:
ids = set(families["parent"])

In [8]:
# ---- Crimes and charges

charges = gather(
    paths.data.dst_raw,
    names=range(1980, 2022),
    file_pattern="krsi{name}.dta",
    concatenate=True,
    columns=["pnr", "sig_sigtdto", "sig_ger1dto"],
    filters={"pnr": ids, "sig_ger7": lambda s: s.between(1000000, 1999999)},
)

# Collapse to monthly level
crimes = make_monthly(charges, date_col="sig_ger1dto", name="crimes")
charges = make_monthly(charges, date_col="sig_sigtdto", name="charges")

In [9]:
# ---- Convictions

conv = gather(
    paths.data.dst_raw,
    names=range(1980, 2022),
    file_pattern="kraf{name}.dta",
    concatenate=True,
    columns=["pnr", "afg_afgoedto"],
    filters={"pnr": ids, "afg_ger7": lambda s: s.between(1000000, 1999999)},
)

# Collapse to monthly level
conv = make_monthly(conv, date_col="afg_afgoedto", name="convictions")

In [10]:
# ---- All incarcerations

incar = gather(
    paths.data.crime / "krin_placering.dta",
    columns=["pnr", "handelse", "fgsldto", "losldto"],
    filters={"pnr": ids},
    concatenate=True,
)

incar = incar.rename(
    columns={"handelse": "type", "fgsldto": "entry", "losldto": "exit"}
)

In [11]:
# ---- Arrests

arrests = incar.query("type <= 2")
arrests = make_monthly(arrests, date_col="entry", name="arrests")

In [12]:
# ---- Non-arrest incarcerations

# Pre-trial custody, serving time, and transfers
incar = incar.query("type >= 3")

# Drop older records with missing release date
incar = incar.loc[lambda d: ~(d["exit"].isna() & d["entry"].dt.year.le(1990))]

# Impute missing transfer dates
mask = incar["exit"].isna() & incar["type"].eq(6)
incar["exit"] = incar["exit"].mask(mask, incar["entry"])

# Recent incarcerations with missing release dates -> assume ongoing
mask = incar["exit"].isna() & incar["entry"].ge(pd.Timestamp("2019-01-01"))
incar["exit"] = incar["exit"].mask(mask, pd.Timestamp("2021-08-01"))

# Drop double registrations
incar = incar.sort_values(
    ["pnr", "entry", "exit"], na_position="first"
).drop_duplicates(["pnr", "entry"], keep="last")

# Set remaining missing release dates to day of incarceration and mark as imputed
incar["imputed"] = incar["exit"].isna()
incar["exit"] = incar["exit"].where(incar["exit"].notna(), incar["entry"])

# ---- Collapse overlapping spells

spells = (
    incar.groupby("pnr", group_keys=False)
    .apply(merge_spells)
    .reset_index(drop=True)
)

# Mark imputed
imputees = incar.query("imputed").pipe(lambda d: set(zip(d["pnr"], d["exit"])))
spells["imputed"] = pd.Series(zip(spells["pnr"], spells["exit"])).isin(imputees)

# Make monthly
spells = spells.assign(
    **{c: lambda d, c=c: d[c].dt.to_period("M") for c in ["entry", "exit"]}
)

# Save entry into incarceration as event
entries = (
    spells[["pnr", "entry"]].rename(columns={"entry": "month"}).assign(entries=1)
).groupby(["pnr", "month"], as_index=False).agg({"entries": "sum"})

# Make incarceration data long
spells = (
    spells.set_index("pnr")
    .apply(lambda r: pd.period_range(r["entry"], r["exit"], freq="M"), axis=1)
    .explode()
    .rename("month")
    .reset_index()
    .drop_duplicates()
)

spells["incarcerated"] = 1

# Gather all records in dictionary
records = {
    "crime": crimes,
    "charge": charges,
    "conviction": conv,
    "arrest": arrests,
    "incarceration": entries,
    "in_prison": spells,
}

### Setting Up Panels

We set up an event time panel for each of our events (crime, arrest, charge, etc.).

This is done for each child by finding the earliest recorded case of a given event from an *active* parent, since the child started school.

We then add these dates to the monthly absence data collected above and generate their respective event time variables.

Note: We restrict all panels to be a subset of the "first parent crime"-panel. I.e., for a child to be in any panel, they have to have experienced a parental crime since starting school.

The panels are stored in a dictionary of DataFrames in `temp/panels.pkl`.

In [13]:
from absence_and_crime.transforms import age_from_months

In [14]:
# in_prison does not contain events but spells
events = {c: records[c] for c in records if c not in ["in_prison"]}

In [15]:
def find_first_event(
    criminal_record: pd.DataFrame, families: pd.DataFrame
) -> pd.DataFrame:
    first_event = (
        families
        # Keep children w. parent w. a given event
        .merge(
            criminal_record[["pnr", "month"]],
            left_on="parent",
            right_on="pnr",
            how="inner",
            suffixes=("", "_y"),
        )
        # Keep only events since starting school
        .loc[lambda d: d["month"] >= d["start_of_school"]]
        # Keep only events caused by active parents
        .loc[lambda d: d["month"].dt.year.between(d["parent_start"], d["parent_end"])]
        .drop(columns=["pnr_y", "parent_start", "parent_end"])
        # Drop children with more than one event within month
        .loc[
            lambda d: ~d.duplicated(["pnr", "month"]).groupby(d["pnr"]).transform("any")
        ]
        # Keep earliest case
        .sort_values(["pnr", "month"])
        .drop_duplicates(["pnr"], keep="first")
        # Clean-up
        .rename(columns={"month": "t0"})
        .reset_index(drop=True)
    )
    return first_event

In [16]:
panels = {
    event_name: (
        find_first_event(
            criminal_record=event_record, families=families
        )  # define t=0
        .merge(absence, on="pnr", how="left")  # add absence records
        .assign(  # define event time
            event_time=lambda d: d["month"].astype("int64")
            - d["t0"].astype("int64")
        )
        # Child must be observed before and after event
        .loc[
            lambda d: d["event_time"]
            .between(-11, -1)
            .groupby(d["pnr"])
            .transform("any")  # observed before
            & d["event_time"]
            .between(0, 11)
            .groupby(d["pnr"])
            .transform("any")  # observed after
        ]
        .reset_index(drop=True)
    )
    for event_name, event_record in events.items()
}

# Restrict other panels to be a subset of the crime panel
ids = set(panels["crime"]["pnr"])

for k, v in panels.items():
    if k != "crime":
        panels[k] = v.loc[lambda d: d.pnr.isin(ids)]

# Add age in years to panels
for k, v in panels.items():
    v["age"] = age_from_months(v.date_of_birth, v.month)

# Save
with open(paths.temp / "panels.pkl", "wb") as f:
    pickle.dump(panels, f)

### Gathering Covariates

We gather the following socio-economic variables for each child
- Total parent earnings (1.000 DKK)
- Mean parent share of fulltime (based on hours worked)
- Parents' combined net assets
- Highest completed among parents

These variables are all defined in the year the child started school.

The resulting data is stored in `temp/covariates.parquet`.

In [17]:
covariates = families.copy()

covariates = (
    covariates
    .assign(
        year=lambda d: d["start_of_school"].dt.year,
        age_at_start=lambda d: d["start_of_school"].dt.year
        - d["date_of_birth"].dt.year
        - (d["start_of_school"].dt.month < d["date_of_birth"].dt.month),
    )
    # Parents from child's first year of school
    .loc[lambda d: d["year"].between(d["parent_start"], d["parent_end"])]
    .assign(
        single_parent=lambda d: d["pnr"].groupby(d["pnr"]).transform("count").eq(1)
    )
    .drop(columns=["parent_start", "parent_end"])
)

In [18]:
ids = (
    covariates[["parent", "year"]]
    .rename(columns={"parent": "pnr"})
    .drop_duplicates()
)
lmo = {}

for t in range(2010, 2022):

    py = ids.loc[covariates["year"].eq(t)]  # This year's parents

    # ---- BFL: Wages and hours worked

    vl = ["pnr", "ajo_smalt_loenbeloeb", "ajo_fuldtid_beskaeftiget"]
    tmp = (
        fetch(paths.data.dst_raw / f"bfl{t}.dta", filters={"pnr": py["pnr"]}, columns=vl)
        .groupby("pnr", as_index=False)
        .agg(  # Collapse by parent
            wages=(
                "ajo_smalt_loenbeloeb",
                lambda x: x.sum() / 1000,
            ),  # Total wages, 1.000 DKK
            fulltime=(
                "ajo_fuldtid_beskaeftiget",
                lambda x: x.sum() / 12,
            ),  # Share of fulltime (monthly)
        )
    )

    # Merge onto parent list
    py = py.merge(tmp, on="pnr", how="left")

    # ---- IND: Assets

    vl = ["pnr", "formrest_ny05"]
    tmp = (
        fetch(paths.data.dst_raw / f"ind{t}.dta", filters={"pnr": py["pnr"]}, columns=vl)
        .groupby("pnr", as_index=False)
        .agg(
            assets=(
                "formrest_ny05",
                lambda x: x.sum() / 1000,
            ),  # Net assets, 1.000 DKK
        )
    )

    # Merge onto parent list
    py = py.merge(tmp, on="pnr", how="left")

    # Assuming missing is 0
    py = py.fillna(0)

    # ---- UDDA: Education

    vl = ["pnr", "hfaudd"]
    tmp = fetch(
        paths.data.dst_raw / f"udda09_{t}.dta", filters={"pnr": py["pnr"]}, columns=vl
    )

    # Merge onto parent list
    py = py.merge(tmp, on="pnr", how="left")

    lmo[t] = py

# Combine all years into one
lmo = pd.concat(lmo.values(), ignore_index=True)

In [19]:
# Merge onto children
covariates = covariates.merge(
    lmo.rename(columns={"pnr": "parent"}),
    on=["parent", "year"],
    how="left",
    validate="m:1",
)

In [20]:
# Truncate
for i in ["wages", "fulltime", "assets"]:
    covariates[i].clip(upper=covariates[i].quantile(0.99), inplace=True)
    if i in ["wages", "fulltime"]:
        covariates[i].clip(lower=0, inplace=True)
    else:
        covariates[i].clip(lower=covariates[i].quantile(0.01), inplace=True)

In [21]:
# Recode educations levels
cats = {  # 4 rough ISCED levels
    "Primary": 1,
    "Lower secondary": 1,  # 1: Grundskole (Pre-High School)
    "Upper secondary": 2,  # 2: Gymnasie (High School)
    "Short cycle tertiary": 3,  # 3: KVU (post-High School, non-college)
    "Bachelor or equivalent": 4,
    "Master or equivalent": 4,
    "Doctoral or equivalent": 4,  # 4: MVU/LVU (Some college degree)
}

mapping = (
    pd.read_stata(paths.data.disced / "c_audd_niveau_e_l1l2_t.dta")
    .assign(
        hfaudd=lambda df: df["start"].astype(int),
        isced=lambda df: df["AUDD_NIVEAU_E_L1L2_T"].map(cats),
    )[["hfaudd", "isced"]]
    .dropna()
    .set_index("hfaudd")["isced"]
    .to_dict()
)

covariates = covariates.assign(education=lambda d: d["hfaudd"].map(mapping)).drop(
    columns=["hfaudd"]
)

In [22]:
# ---- Collapse by child

cons = ["date_of_birth", "start_of_school", "age_at_start", "single_parent"]
dkk = ["wages", "assets"]

covariates = covariates.groupby("pnr", as_index=False).agg(
    {
        **{c: "last" for c in cons},  # child-specific variables
        **{c: "sum" for c in dkk},  # total wages, transfers and net asset
        "fulltime": "mean",  # mean share of fulltime
        "education": "max",  # highest level of education among parents
    }
)

# ---- Add child demographics

covariates = covariates.merge(  # add origin and sex
    families[["pnr", "non_danish", "female"]].drop_duplicates(),
    on=["pnr"],
    how="left",
    validate="1:1",
)

# Save
covariates.to_parquet(paths.temp / "covariates.parquet")

### Identifying Household Instability

We gather the following measures of household instability:
- Child address changes
- Parent job loss
- Parent substance abuse
- Parent psychiatric diagnosis

The definitions of the above events are defined in detail in the paper.

These are stored as event data (id, month) in a dictionary of DataFrames across instability measure and saved to `temp/instability.pkl`.

In [23]:
child_ids = set(panels["crime"]["pnr"])

parent_ids = set(
    families[["pnr", "parent", "parent_start", "parent_end"]]
    .merge(panels["crime"][["pnr", "t0"]].drop_duplicates(), on="pnr", how="inner")
    .loc[
        lambda d: d["t0"].dt.year.between(d["parent_start"], d["parent_end"]),
        "parent",
    ]
)

instability = {}

In [24]:
# ---- Address changes

instability["address"] = (
    fetch(
        paths.data.dst_raw / "befbop12_2021.dta",
        columns=["pnr", "bop_vfra"],
        filters={"pnr": child_ids},
    )
    .assign(
        month=lambda d: pd.to_datetime(d["bop_vfra"], errors="coerce").dt.to_period(
            "M"
        )
    )
    .dropna(subset=["month"])[["pnr", "month"]]
    .drop_duplicates()
)

In [25]:
# ---- Job Loss

# BFL: Wages and hours worked
job = gather(
    paths.data.dst_raw,
    names=range(2008, 2022),
    file_pattern="bfl{name}.dta",
    concatenate=True,
    columns=[
        "pnr",
        "ajo_smalt_loenbeloeb",
        "ajo_fuldtid_beskaeftiget",
        "ajo_job_slut_dato",
    ],
    filters={"pnr": parent_ids},
)

# Make monthly
job = (
    job.assign(month=job["ajo_job_slut_dato"].dt.to_period("M"))
    .groupby(["pnr", "month"], as_index=False)
    .agg(
        wages=("ajo_smalt_loenbeloeb", lambda x: x.sum()),  # Wages, DKK
        fulltime=(
            "ajo_fuldtid_beskaeftiget",
            lambda x: x.sum(),
        ),  # Share of fulltime (monthly)
    )
)

job = job.query("fulltime > 0")[["pnr", "month"]]

obsend = pd.Period("2021-12", freq="M")

# Compute next month and check presence
instability["jobloss"] = (
    job[["pnr"]]
    # Next month: Month after recorded employment
    .assign(month=job["month"] + 1)
    # Still employed?
    .merge(job, on=["pnr", "month"], how="left", indicator=True)
    # Keep only job-losses
    .loc[lambda d: d["month"].le(obsend) & d["_merge"].eq("left_only")]
    # Sort and clean
    .drop(columns=["_merge"])
    .sort_values(["pnr", "month"])
    .reset_index(drop=True)
)

del job

In [26]:
# ---- Parent substance abuse (via diagnoses and prescriptions)

# Diagnoses, somatic hospitals
adm = gather(  # Administrative data w. patient IDs
    paths.data.dst_raw,
    names=range(2010, 2019),
    file_pattern="lpr_adm{name}.dta",
    concatenate=True,
    columns=["pnr", "recnum", "c_kontaars", "d_inddto"],
    filters={"pnr": parent_ids},
)

codes = gather(  # Link between patient IDs and diagnoses
    paths.data.dst_raw,
    names=range(2010, 2019),
    file_pattern="lpr_diag{name}.dta",
    concatenate=True,
    columns=["recnum", "c_diag", "c_diagtype"],
    filters={"recnum": adm["recnum"]},
)

diag = adm.merge(codes, on="recnum", how="left").drop(columns=["recnum"])

# Add psychiatry
diag = pd.concat(
    [
        # LPR_DIAG
        diag.assign(psyk=0),
        # LPR_PSYK
        fetch(
            paths.data.dst_raw / "psyk_adm2018.dta",
            columns=["pnr", "c_adiag", "d_inddto", "c_kontaars"],
            filters={"pnr": parent_ids},
        )
        .assign(psyk=1)
        .rename(columns={"c_adiag": "c_diag"}),
    ],
    ignore_index=True,
)

del adm, codes

# Prescriptions
lmdb = {
    y: (
        pd.read_sas(paths.data.dst_raw / f"lmdb{y}.sas7bdat", encoding="latin1")[
            ["PNR", "eksd", "atc3"]
        ]
        .assign(pnr=lambda d: pd.to_numeric(d["PNR"], errors="coerce"))
        .drop(columns=["PNR"])
        .loc[
            lambda d: d["pnr"].isin(parent_ids)
            & d["atc3"].str.startswith("N", na=False)
        ]
    )
    for y in range(2008, 2022)
}

lmdb = pd.concat(lmdb.values(), ignore_index=True)

# From diagnoses
s1 = diag.loc[lambda d: d["c_diag"].str.startswith("DF1", na=False)].assign(
    month=lambda d: pd.to_datetime(d["d_inddto"], errors="coerce").dt.to_period("M")
)[["pnr", "month"]]

# From prescriptions
s2 = lmdb.loc[lambda d: d["atc3"].str.startswith("N07B", na=False)].assign(
    month=lambda d: pd.to_datetime(d["eksd"], errors="coerce").dt.to_period("M")
)[["pnr", "month"]]

instability["substance"] = (
    pd.concat([s1, s2], ignore_index=True).drop_duplicates().reset_index(drop=True)
)

del s1, s2, lmdb

In [27]:
# ---- Parent psych. diagnosis

selfharm = [f"DX{i}" for i in range(60, 85)]
damage = ["DS51", "DS55", "DS59", "DS61", "DS65", "DS69"]
damage = damage + [f"DT{i}" for i in range(36, 61) if i != 51]

instability["mental"] = (
    diag.assign(
        # In somatic hospital w. psychiatric diagnosis
        psychiatric=lambda d: d["c_diag"].str.startswith("DF", na=False)
        & d["c_diagtype"].eq("A")
        | d["psyk"],
        # Attempted suicide or self harm
        selfharm=lambda d: d["c_kontaars"].eq("4")
        | d["c_diag"].str[0:4].isin(selfharm),
        # Damage or poisoning as supplemental diagnosis
        damage=lambda d: d["c_diagtype"].ne("A")
        & d["c_diag"].str[0:4].isin(damage)
        & ~d["psyk"],
    )
    .assign(
        contact=lambda d: d["psychiatric"] | d["selfharm"],
        month=lambda d: d["d_inddto"].dt.to_period("M"),
    )
    .loc[lambda d: d["contact"], ["pnr", "month"]]
    .drop_duplicates()
)

# Force order to fit paper
instability = {k: instability[k] for k in ["address", "jobloss", "mental", "substance"]}

# Save
with open(paths.temp / "instability.pkl", "wb") as f:
    pickle.dump(instability, f)